# 01_standardize.ipynb
## Mục tiêu
Đây là giai đoạn đầu tiên trong quá trình processing dữ liệu thô. Những đoạn code sau sẽ làm những nhiệm vụ:
1. Parse các file json trong dataset.
2. Bổ xung thêm cột liên quan đến dự án.
3. Gộp mọi thứ lại trong một file csv để dễ đọc trong các quy trình sau.

## 1. Import library
Import các library như pandas và StringIO để đọc dữ liệu raw, đồng thời import requests để lấy thêm metadata từ các repo.

In [ ]:
import requests
import pandas as pd

import time
from datetime import datetime

import os
from io import StringIO
from concurrent.futures import ThreadPoolExecutor

## 2. Đọc dữ liệu
Load toàn bộ dataset vào memory sử dụng pandas.

In [ ]:
DATASET_PATH = "../data/raw/comment_codeblock_dataset.jsonl"
DATASET_WITH_CHANGE_PATH = "../data/raw/comment_codeblock_first_change_dataset.jsonl"

full_dataset = pd.read_json(DATASET_PATH, lines=True)
dataset_with_change = pd.read_json(DATASET_WITH_CHANGE_PATH, lines=True)

dataset = full_dataset.merge(dataset_with_change, how="left")

print(dataset.head())
print(dataset_with_change.head())

## 3. Merge datasets vào bảng dữ liệu
Define bảng dữ liệu output. Xử lý dataset gốc và chuyển nó thành bảng dữ liệu của riêng mình.

In [ ]:
output = pd.DataFrame()

TOKEN = os.getenv("GITHUB_TOKEN")
if not TOKEN:
    raise RuntimeError(
        "Please set the GITHUB_TOKEN environment variable before running this cell."
    )

session = requests.Session()
session.headers.update(
    {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json",
    }
)

def blob_to_commit(blob):
    parts = blob.split("/")
    return f"https://github.com/{parts[3]}/{parts[4]}/commit/{parts[6]}"


def fetch_graphql_batch(batch):
    query_parts = []
    for idx, (owner, repo, sha) in enumerate(batch):
        query_parts.append(
            f'c{idx}: repository(owner: "{owner}", name: "{repo}") {{ object(expression: "{sha}") {{ ... on Commit {{ committedDate }} }} }}'
        )

    query = f"query {{ {' '.join(query_parts)} }}"

    try:
        r = session.post(
            "https://api.github.com/graphql", json={"query": query}, timeout=30
        )
        if r.status_code != 200:
            return [None] * len(batch)

        data = r.json().get("data", {})
        results = []
        for idx in range(len(batch)):
            repo_data = data.get(f"c{idx}") or {}
            obj = repo_data.get("object") or {}
            date = obj.get("committedDate")
            results.append(
                pd.to_datetime(date, utc=True).strftime("%Y-%m-%dT%H:%M:%SZ")
                if date
                else None
            )
        return results
    except requests.RequestException:
        return [None] * len(batch)


def get_all_commit_dates(commit_urls, batch_size=50):
    parsed_all = []
    for url in commit_urls:
        parts = url.split("/")
        parsed_all.append((parts[3], parts[4], parts[6]))

    unique = list(dict.fromkeys(parsed_all))  # dedupe, preserve order
    batches = [unique[i:i + batch_size] for i in range(0, len(unique), batch_size)]

    date_map = {}
    with ThreadPoolExecutor(max_workers=10) as executor:
        for batch, batch_results in zip(batches, executor.map(fetch_graphql_batch, batches)):
            for key, date in zip(batch, batch_results):
                date_map[key] = date

    return [date_map.get(key) for key in parsed_all]
    
output["id"] = dataset["match_id"]
output["repo"] = dataset["repo_full_name"]
output["language"] = dataset["language"]
output["commit_url"] = dataset["file_url"].apply(blob_to_commit)
output["file"] = dataset["path"]
output["code_block"] = dataset["code_block"].apply(lambda block : block.replace("\n", "\\n"))
output["commit_date"] = get_all_commit_dates(output["commit_url"])
output["change_commit_url"] = dataset["first_change_commit_url"]
output["change_commit_date"] = dataset["first_change_date"]

print(output.head())

## 4. Export bảng dữ liệu
Export vào folder interim trong data để tiếp tục clean.

In [ ]:
INTERIM_DIR = "../data/interim"
if not os.path.exists(INTERIM_DIR):
    os.makedirs(INTERIM_DIR)
    
PROCESSED_DATA_PATH = f"{INTERIM_DIR}/standardized_data_{datetime.today().strftime('%Y-%m-%d %H:%M:%S')}.csv"
output.to_csv(PROCESSED_DATA_PATH, index=False)